# Project SENTINEL — RFdiffusion backbone generation (GPU, one-click)

This notebook runs the **full-scale GPU step** that this project's CPU
sandbox could not run locally (RFdiffusion needs a CUDA SE3-Transformer —
see `results/design/GPU_TIER_STATUS.md` in the repo for why, and
`src/sentinel/design/backbone_gen.py` for the CPU geometric baseline that
was used instead so the rest of the pipeline could run end-to-end).

**What this does:** generates real, RFdiffusion-designed protein backbones
conditioned on the exact AD-fold hotspot residues identified in Year 1's
strain fingerprint (`results/atlas/ad_strain_fingerprint.json`), targeting
PDB **5O3L** (AD_PHF),
core residues 306-378.

**How to use:** Runtime -> Change runtime type -> GPU (T4 is fine, free
tier). Then Runtime -> Run all. Takes ~15-30 min. Output backbones download
as a zip; drop the .pdb files into `results/design/backbones/rfdiffusion/`
in the repo and re-run `make design` — the active-learning loop reads
backbones from that directory with no code changes needed.


In [ ]:
#@title 1. Install RFdiffusion (~5 min)
!git clone https://github.com/RosettaCommons/RFdiffusion.git
%cd RFdiffusion
# Three real, verified bugs found and fixed here in sequence (not guessed -- see
# PROGRESS_LOG.md): (1) se3-transformer isn't on PyPI, only the bundled
# env/SE3Transformer/ subpackage; (2) DGL has no published wheel matching Colab's
# default torch/CUDA build, fixed by pinning torch to DGL's actual supported CUDA
# 12.1 tag; (3) DGL 2.x's package __init__ unconditionally imports its (unused by
# RFdiffusion) graphbolt submodule, which needs torchdata -- not installed by
# default -- causing "from torchdata.datapipes.iter import IterDataPipe" to fail.
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q dgl -f https://data.dgl.ai/wheels/cu121/repo.html
!pip install -q torchdata hydra-core pyrsistent icecream decorator pynvml
!pip install -q -e env/SE3Transformer
!pip install -q -e . --no-deps
!mkdir -p models && cd models && \
  wget -q http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt


In [ ]:
#@title 2. Fetch the AD-fold target structure and set hotspots (pre-filled from Year-1 results)
!wget -q https://files.rcsb.org/download/5O3L.pdb -O target.pdb

CONTIG = "A306-378/0 60-120"  #@param {type:"string"}
HOTSPOT_RES = "A365,A357,A342,A326,A315,A354,A334,A323,A361,A313,A346,A310,A362,A320,A374"  #@param {type:"string"}
NUM_DESIGNS = 128  #@param {type:"integer"}
print("contig:", CONTIG)
print("hotspots:", HOTSPOT_RES)


In [ ]:
#@title 3. Run RFdiffusion
!python scripts/run_inference.py \
    inference.output_prefix=outputs/ad_capper \
    inference.input_pdb=../target.pdb \
    'contigmap.contigs=[$CONTIG]' \
    "ppi.hotspot_res=[$HOTSPOT_RES]" \
    inference.num_designs=$NUM_DESIGNS


In [ ]:
#@title 4. Zip and download results
import shutil
from google.colab import files
shutil.make_archive("rfdiffusion_backbones", "zip", "outputs")
files.download("rfdiffusion_backbones.zip")
print("Unzip into results/design/backbones/rfdiffusion/ in the repo, then: make design")
